# 01 — Exploratory Data Analysis
**Inventory Intelligence System | Uninox Houseware**

Analyse demand patterns, stock levels, supplier performance, and seasonal trends.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
from config import Paths
demand   = pd.read_csv(Paths.DATA_RAW / 'demand_history.csv')
inventory = pd.read_csv(Paths.DATA_RAW / 'inventory_history.csv')
po       = pd.read_csv(Paths.DATA_RAW / 'purchase_orders.csv')
invoices = pd.read_csv(Paths.DATA_RAW / 'sales_invoices.csv')
terms    = pd.read_csv(Paths.DATA_RAW / 'supplier_terms.csv')
direc    = pd.read_csv(Paths.DATA_RAW / 'supplier_directory.csv')
print(f'demand: {demand.shape}  inventory: {inventory.shape}  POs: {po.shape}')

## 1. Demand Distribution by SKU

In [ ]:
sku_summary = demand.groupby('sku_id').agg(
    total_units=('net_units','sum'),
    avg_monthly=('net_units','mean'),
    std_dev=('net_units','std'),
    periods=('period','count')
).reset_index().sort_values('total_units', ascending=False)
print(sku_summary.to_string())
fig = px.bar(sku_summary, x='sku_id', y='avg_monthly', color='avg_monthly',
             title='Average Monthly Demand by SKU', template='plotly_dark')
fig.show()

## 2. ABC Classification Check

In [ ]:
abc = demand.groupby(['sku_id','abc_class'])['net_units'].sum().reset_index()
fig2 = px.pie(demand.drop_duplicates('sku_id'), names='abc_class',
              title='SKU Count by ABC Class', template='plotly_dark')
fig2.show()

## 3. Demand Time Series — Top 5 SKUs

In [ ]:
demand['period_dt'] = pd.to_datetime(demand['period'], format='%Y-%m')
top5 = sku_summary.head(5)['sku_id'].tolist()
fig3 = px.line(demand[demand['sku_id'].isin(top5)], x='period_dt', y='net_units',
               color='sku_id', title='Monthly Demand — Top 5 SKUs', template='plotly_dark')
fig3.show()

## 4. Inventory Status: Stock vs ROP

In [ ]:
fig4 = go.Figure()
fig4.add_bar(x=inventory['sku_id'], y=inventory['total_available'], name='Available', marker_color='#4B8BBE')
fig4.add_bar(x=inventory['sku_id'], y=inventory['reorder_point'], name='Reorder Point', marker_color='#e74c3c', opacity=0.7)
fig4.update_layout(barmode='overlay', template='plotly_dark', title='Stock Available vs Reorder Point', xaxis_tickangle=-45)
fig4.show()
alerts = inventory[inventory['total_available'] <= inventory['reorder_point']]
print(f'SKUs at or below ROP: {len(alerts)}')
print(alerts[['sku_id','item_name','total_available','reorder_point','status']])

## 5. Supplier Performance

In [ ]:
sup_perf = direc.sort_values('on_time_rate_pct', ascending=False)
fig5 = px.bar(sup_perf, x='supplier_name', y='on_time_rate_pct', color='on_time_rate_pct',
              color_continuous_scale='RdYlGn', title='Supplier On-Time Rate (%)',
              template='plotly_dark')
fig5.update_xaxes(tickangle=-30)
fig5.show()

## 6. Missing Values & Data Quality

In [ ]:
for name, df in [('demand',demand),('inventory',inventory),('PO',po)]:
    missing = df.isnull().sum().sum()
    print(f'{name:12s}: {len(df):5d} rows, {missing} missing values ({missing/df.size*100:.1f}%)')